<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW11 · Session 2 · Scaling, Dates, Encoding, and Inconsistent Data

        **Course level:** developing beginner  
        **Prior learning:** quality profiling, missing-data decisions, and pandas transformations  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        After profiling, we standardise representations while preserving meaning. The session covers messy labels, mixed dates, invalid ranges, encodings, and numerical scaling.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - standardise text categories with explicit mappings
- parse mixed-format dates and identify failures
- repair invalid values using documented rules
- compare standard, min-max, and robust scaling
- validate a cleaned dataset with assertions


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load a fresh copy

Starting from the source makes the workflow reproducible and prevents hidden dependence on Session 1.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

raw = pd.read_csv(DATA_DIR / "business_sales_dirty.csv")
data = raw.copy()
print(data.shape)


## 1. Standardise inconsistent text

Use a transparent sequence: choose pandas string type, trim whitespace, standardise case, then map known synonyms. Do not use approximate matching without review when categories have important consequences.


### Clean region labels

Missing values are filled after normalising valid strings.


In [ ]:
data["region"] = (
    data["region"]
    .astype("string")
    .str.strip()
    .str.title()
    .fillna("Unknown")
)
print(data["region"].value_counts(dropna=False))


### Map channel synonyms

An explicit dictionary documents that telephone and phone mean the same category.


In [ ]:
data["channel"] = data["channel"].astype("string").str.strip().str.title()
channel_map = {"Telephone": "Phone"}
data["channel"] = data["channel"].replace(channel_map)

print(data["channel"].value_counts(dropna=False))


## 2. Parse mixed-format dates

Date strings may be ambiguous. State assumptions such as day-first order, coerce invalid values to missing timestamps, and inspect every failure.


### Parse and audit dates

format='mixed' lets pandas infer each element, while dayfirst=True documents how 03/04 is interpreted.


In [ ]:
data["order_date_parsed"] = pd.to_datetime(
    data["order_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)
failed_dates = data.loc[
    data["order_date_parsed"].isna(),
    ["order_id", "order_date"],
]

print("Failed dates:", len(failed_dates))
print(failed_dates)


## 3. Repair invalid numerical values

Correction requires a defensible rule. If the source cannot be corrected, mark impossible values as missing and then apply the documented missing-data strategy.


### Convert impossible price and unit values to missing

Mask changes only rows that fail the stated rules.


In [ ]:
data["unit_price"] = data["unit_price"].mask(data["unit_price"] <= 0)
data["units"] = data["units"].mask(~data["units"].between(1, 20))

print("Missing prices after rule:", data["unit_price"].isna().sum())
print("Missing units after rule:", data["units"].isna().sum())


### Fill values after defining invalidity

Median filling is shown for teaching. In production, consult data owners and test sensitivity to the decision.


In [ ]:
data["units_missing"] = data["units"].isna()
data["price_missing"] = data["unit_price"].isna()
data["units"] = data["units"].fillna(data.groupby("category")["units"].transform("median"))
data["unit_price"] = data["unit_price"].fillna(data.groupby("category")["unit_price"].transform("median"))
data["satisfaction"] = data["satisfaction"].fillna(data["satisfaction"].median())

print(data[["units", "unit_price", "satisfaction"]].isna().sum())


## 4. Character encodings

Encoding problems happen before analysis. Read bytes using the documented source encoding and convert to Unicode strings in memory. UTF-8 is a good exchange format for new outputs.


### Decode Latin-1 correctly

Trying UTF-8 on this teaching file would raise a decoding error. Latin-1 preserves the original characters.


In [ ]:
encoded_path = DATA_DIR / "sample_notes_latin1.txt"
decoded_text = encoded_path.read_text(encoding="latin-1")
utf8_bytes = decoded_text.encode("utf-8")

print(decoded_text)
print("UTF-8 byte length:", len(utf8_bytes))


## 5. Scale numerical features

Scaling changes representation, not the underlying ranking.

- StandardScaler: mean approximately 0 and standard deviation approximately 1.
- MinMaxScaler: maps the training range to 0 through 1.
- RobustScaler: centres by median and scales by IQR, reducing outlier influence.

Fit scalers on training data only in machine learning. Here we compare methods descriptively.


### Compare three scaling methods

The same two features receive three transformations. Column names retain the method and source.


In [ ]:
numeric_features = data[["units", "unit_price"]]
standard_values = StandardScaler().fit_transform(numeric_features)
minmax_values = MinMaxScaler().fit_transform(numeric_features)
robust_values = RobustScaler().fit_transform(numeric_features)

scaled_preview = pd.DataFrame({
    "units_original": numeric_features["units"],
    "units_standard": standard_values[:, 0],
    "units_minmax": minmax_values[:, 0],
    "units_robust": robust_values[:, 0],
})
print(scaled_preview.head().round(3))


## 6. Encode categories for numerical models

One-hot encoding creates an indicator column per category. Preserve the original category for reporting, and avoid fitting the category vocabulary on future test data.


### Create one-hot columns

dtype=int makes the indicators easy to inspect.


In [ ]:
channel_indicators = pd.get_dummies(
    data["channel"],
    prefix="channel",
    dtype=int,
)
print(channel_indicators.head())


## Guided cleaning pipeline

Order matters: remove exact duplicates, standardise text, parse dates, define invalid values, treat missingness, engineer revenue, and validate.


### Create and validate a clean analysis table

Assertions express the final data contract.


In [ ]:
final_data = data.drop_duplicates().copy()
final_data = final_data.dropna(subset=["order_date_parsed"]).copy()
final_data["net_revenue"] = (
    final_data["units"]
    * final_data["unit_price"]
    * (1 - final_data["discount"])
)

assert final_data["order_id"].is_unique
assert final_data["order_date_parsed"].notna().all()
assert final_data["units"].between(1, 20).all()
assert (final_data["unit_price"] > 0).all()
assert final_data["region"].isin(["North", "South", "East", "West", "Unknown"]).all()
assert (final_data["net_revenue"] >= 0).all()

print("Validated shape:", final_data.shape)
print(final_data[["order_id", "order_date_parsed", "region", "net_revenue"]].head())


## Student coding lab

Start from a fresh copy for each task so that one experiment does not hide another.


        ### Student task 1: Standardise categories

        <div class="task"><strong>Your job:</strong> Clean region and channel labels, map Telephone to Phone, and print sorted category counts including missing values.</div>

        **Check your work**

        - Whitespace is removed.
- Capitalisation is consistent.
- Telephone is mapped explicitly rather than silently guessed.


In [ ]:
# STUDENT TASK 1
text_cleaned = raw.copy()
# Add text cleaning and the explicit mapping.
print(text_cleaned["region"].value_counts(dropna=False))
print(text_cleaned["channel"].value_counts(dropna=False))


        ### Student task 2: Parse dates safely

        <div class="task"><strong>Your job:</strong> Parse order_date with mixed formats, day-first interpretation, and coercion. Print the failure count and original failed values.</div>

        **Check your work**

        - Valid dates become datetime values.
- Invalid text becomes NaT.
- Original strings remain available for investigation.


In [ ]:
# STUDENT TASK 2
date_data = raw[["order_id", "order_date"]].copy()
date_data["parsed_date"] = pd.NaT
failed_date_rows = date_data[date_data["parsed_date"].isna()]
print("Failures:", len(failed_date_rows))
print(failed_date_rows.head())


        ### Student task 3: Repair invalid ranges

        <div class="task"><strong>Your job:</strong> Mark non-positive prices and units outside 1 to 20 as missing. Add Boolean indicators first, then fill each using its category median.</div>

        **Check your work**

        - Indicators reflect original or rule-created missingness.
- Only invalid values are masked.
- No units or prices remain missing after a justified fill.


In [ ]:
# STUDENT TASK 3
repaired = raw.copy()
# Add indicators, masking, and grouped median filling.
print(repaired[["units", "unit_price"]].isna().sum())


        ### Student task 4: Compare scalers

        <div class="task"><strong>Your job:</strong> Scale marketing_spend with StandardScaler, MinMaxScaler, and RobustScaler. Report min, median, mean, max, and standard deviation for each result.</div>

        **Check your work**

        - Each scaler is fitted to a two-dimensional input.
- Three transformed columns are compared.
- The summaries reveal how the methods differ.


In [ ]:
# STUDENT TASK 4
scale_source = raw[["marketing_spend"]].dropna()
scaling_summary = pd.DataFrame()
print(scaling_summary)


        ### Student task 5: Write a final data contract

        <div class="task"><strong>Your job:</strong> Create a cleaned copy and add at least six assertions covering unique order IDs, valid dates, recognised regions/channels, valid numerical ranges, and non-negative calculated revenue.</div>

        **Check your work**

        - The table is actually cleaned before it is asserted.
- Each assertion has a useful message.
- The supplied dataset passes all checks.



**Optional extension:** Wrap the transformations in a function that returns both the clean table and a quality report.


In [ ]:
# STUDENT TASK 5
contract_table = raw.copy()
# Apply the necessary cleaning before the assertions.
# Add at least six assertions here.
print("Rows ready for analysis:", len(contract_table))


        ## Knowledge check

        Answer these without running new code first.

        1. Why use explicit mappings for category synonyms?
2. What does errors='coerce' achieve during date parsing?
3. Why define invalid values before filling missingness?
4. How do standard and robust scaling differ?
5. Why must a machine-learning scaler be fitted on training data only?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** TW12 turns clean data into honest bar, line, histogram, scatter, stacked, and box-plot stories.
